# Day 13: CNN–ViT Hybrid Fusion Network 🛰️

## 1. Learning Objective & Milestone Scope

On **Day 8**, we constructed the baseline **3D-CNN** (`Hyperspectral3DCNN`), capturing local spatial-spectral textures with 3D convolution kernels. On **Day 12**, we implemented the **Vision Transformer** (`HyperspectralViT`), capturing global pairwise spatial affinities with multi-head self-attention.

Today (**Day 13**), we unite these two complementary paradigms into a single unified architecture: the **CNN–ViT Hybrid Fusion Network** (`HyperspectralHybridCNNViT`).

### Why Hybrid Fusion for Hyperspectral Imaging?
1. **Complementary Inductive Biases**:
   - **3D-CNN Branch**: Possesses strong translational equivariance and local neighborhood bias. Excels at detecting subtle spectral absorption dips and local edge transitions within small sub-windows ($3 \times 3 \times 3$).
   - **ViT Branch**: Possesses global receptive fields without locality restrictions. Excels at modeling long-range contextual relationships and cross-canopy correlations across the full $7 \times 7$ patch ($49 \times 49$ attention matrix).
2. **Unified Representation**:
   - Rather than relying on a single inductive bias, feature concatenation followed by a learnable projection creates a rich latent feature space combining both local texture and global context.

> [!IMPORTANT]
> **Strict Day 13 Scope**: Today is dedicated solely to **architectural implementation, dimensional verification, and forward-pass validation**. The model is **untrained**. Model training, hyperparameter tuning, and comparative accuracy benchmarks are reserved for Day 14.

## 2. Environment Setup & Model Imports

We import `HyperspectralHybridCNNViT`, `HyperspectralHybridConfig`, and the factory `build_hybrid` from `terraspectra.models`.

In [1]:
import sys
from pathlib import Path

# Resolve project root dynamically
current_dir = Path.cwd().resolve()
if (current_dir / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from terraspectra.models.cnn3d import Hyperspectral3DCNN, Hyperspectral3DCNNConfig
from terraspectra.models.vit import HyperspectralViT, HyperspectralViTConfig
from terraspectra.models.hybrid import (
    HyperspectralHybridCNNViT,
    HyperspectralHybridConfig,
    build_hybrid,
)

print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

PyTorch Version: 2.14.1+cpu
Device: CPU


## 3. Hybrid Architecture & Information Flow

The architecture processes 5D hyperspectral patch tensors $(B, C, D, H, W)$ in parallel through both branches:

```text
               Input Hyperspectral Patch X (B, 1, 15, 7, 7)
                                │
        ┌───────────────────────┴───────────────────────┐
        ▼                                               ▼
  [3D-CNN Branch]                             [ViT Branch]
3D Conv Stages (8->16->32)                  Patch Tokenizer (15 -> 64)
Global Adaptive Pooling (1,1,1)             Positional Embeddings (49 tokens)
Flatten to 1D                               Transformer Encoder Blocks (L=2)
        │                                   LayerNorm & Mean Pooling
        ▼                                               ▼
CNN Feature Vector                          ViT Feature Vector
   (B, 32)                                     (B, 64)
        │                                               │
        └───────────────────────┬───────────────────────┘
                                ▼
                    Feature Concatenation
                           (B, 96)
                                │
                                ▼
                    Fusion Projection Layer
               Linear(96 -> 64) + LayerNorm(64) + GELU() + Dropout(0.2)
                                │
                                ▼
                    Unified Fused Latent Vector
                           (B, 64)
                                │
                                ▼
                     Final Classification Head
                         Linear(64 -> 16)
                                │
                                ▼
                      Output Logits (B, 16)
```

## 4. Feature Extraction & Dimension Alignment

Let's inspect the exact feature representations extracted from both branches before fusion:

- **CNN Feature Vector**: $D_{\text{cnn}} = 32$ (the final convolutional channel count after `AdaptiveAvgPool3d((1, 1, 1))` and flattening).
- **ViT Feature Vector**: $D_{\text{vit}} = 64$ (the latent token embedding dimension after Pre-LayerNorm and spatial mean pooling).
- **Raw Concatenated Vector**: $D_{\text{raw}} = 32 + 64 = 96$.
- **Projected Fused Vector**: $D_{\text{fusion}} = 64$ (projected via `Linear(96, 64)`, normalized via `LayerNorm(64)`, and activated via `GELU`).

In [2]:
# Instantiate hybrid model with Salinas configuration
config = HyperspectralHybridConfig(
    in_channels=1,
    spectral_depth=15,
    spatial_size=7,
    num_classes=16,
    cnn_conv_channels=(8, 16, 32),
    vit_embed_dim=64,
    vit_num_heads=4,
    vit_num_layers=2,
    fusion_dim=64,
    dropout_rate=0.2,
)

hybrid_model = HyperspectralHybridCNNViT(config)
hybrid_model.eval()

print("Model instantiated successfully.")
print(f"  CNN feature dimension:       {hybrid_model.cnn_feature_dim}")
print(f"  ViT feature dimension:       {hybrid_model.vit_feature_dim}")
print(f"  Raw concatenated dimension:  {hybrid_model.raw_fusion_dim}")
print(f"  Projected fusion dimension:  {config.fusion_dim}")
print(f"  Classification classes:      {config.num_classes}")

Model instantiated successfully.
  CNN feature dimension:       32
  ViT feature dimension:       64
  Raw concatenated dimension:  96
  Projected fusion dimension:  64
  Classification classes:      16


## 5. Parameter Counts & Component Breakdown

Let's compute the parameter count across each sub-component of the network:

In [3]:
params = hybrid_model.count_parameters()

print(f"Total Model Parameters:     {params['total_parameters']:,}")
print(f"Trainable Parameters:       {params['trainable_parameters']:,}")
print(f"Non-Trainable Parameters:   {params['non_trainable_parameters']:,}")
print("\nPer-Component Breakdown:")
for k, v in params["component_breakdown"].items():
    pct = (v / params['total_parameters']) * 100
    print(f"  - {k:<30}: {v:>7,} ({pct:>5.1f}%)")

Total Model Parameters:     103,000
Trainable Parameters:       103,000
Non-Trainable Parameters:   0

Per-Component Breakdown:
  - cnn_branch_parameters         :  23,352 ( 22.7%)
  - vit_branch_parameters         :  72,272 ( 70.2%)
  - fusion_projection_parameters  :   6,336 (  6.2%)
  - classifier_parameters         :   1,040 (  1.0%)


## 6. Forward-Pass Demonstration & Dimensional Sanity Checks

We test the forward pass with both a single sample ($B=1$) and a standard batch ($B=8$) using synthetic hyperspectral patches with Salinas-compatible dimensions $(B, 1, 15, 7, 7)$.

In [4]:
# Batch size 1 (single-sample inference check)
x_single = torch.randn(1, 1, 15, 7, 7)
with torch.inference_mode():
    logits_single = hybrid_model(x_single)

print(f"Single-sample input shape:  {x_single.shape}")
print(f"Single-sample logits shape: {logits_single.shape}")
assert logits_single.shape == (1, 16)
assert torch.isfinite(logits_single).all()
print("[SUCCESS] Single-sample inference verified with LayerNorm stability.")

# Batch size 8 (batched processing check with feature inspection)
batch_size = 8
x_batch = torch.randn(batch_size, 1, 15, 7, 7)
with torch.inference_mode():
    logits_batch, feature_dict = hybrid_model(x_batch, return_features=True)

print(f"\nBatched input shape:         {x_batch.shape}")
print(f"CNN feature vector shape:    {feature_dict['cnn_features'].shape} (Expected: (8, 32))")
print(f"ViT feature vector shape:    {feature_dict['vit_features'].shape} (Expected: (8, 64))")
print(f"Fused feature vector shape:  {feature_dict['fused_features'].shape} (Expected: (8, 64))")
print(f"Output logits shape:         {logits_batch.shape} (Expected: (8, 16))")
assert torch.isfinite(logits_batch).all()
print("[SUCCESS] Batched forward pass verified.")

Single-sample input shape:  torch.Size([1, 1, 15, 7, 7])
Single-sample logits shape: torch.Size([1, 16])
[SUCCESS] Single-sample inference verified with LayerNorm stability.

Batched input shape:         torch.Size([8, 1, 15, 7, 7])
CNN feature vector shape:    torch.Size([8, 32]) (Expected: (8, 32))
ViT feature vector shape:    torch.Size([8, 64]) (Expected: (8, 64))
Fused feature vector shape:  torch.Size([8, 64]) (Expected: (8, 64))
Output logits shape:         torch.Size([8, 16]) (Expected: (8, 16))
[SUCCESS] Batched forward pass verified.


## 7. Gradient Backpropagation Verification

To guarantee that the hybrid architecture is fully trainable and ready for Day 14's supervised training loop, we perform a gradient backpropagation smoke test.

In [5]:
hybrid_model.train()

x_train = torch.randn(4, 1, 15, 7, 7)
targets = torch.tensor([0, 5, 8, 15], dtype=torch.long)

criterion = nn.CrossEntropyLoss()
logits = hybrid_model(x_train)
loss = criterion(logits, targets)
loss.backward()

# Verify gradients reach all branches
cnn_grad = hybrid_model.cnn_branch.features[0].weight.grad
vit_grad = hybrid_model.vit_branch.tokenizer.projection.weight.grad
fusion_grad = hybrid_model.fusion_projection[0].weight.grad
head_grad = hybrid_model.classifier.weight.grad

print("Gradient backpropagation check:")
print(f"  CNN branch earliest conv layer grad:      {cnn_grad is not None} (norm: {cnn_grad.norm().item():.4e})")
print(f"  ViT branch tokenizer projection grad:     {vit_grad is not None} (norm: {vit_grad.norm().item():.4e})")
print(f"  Fusion projection linear grad:            {fusion_grad is not None} (norm: {fusion_grad.norm().item():.4e})")
print(f"  Classification head linear grad:          {head_grad is not None} (norm: {head_grad.norm().item():.4e})")

assert cnn_grad is not None and vit_grad is not None and fusion_grad is not None and head_grad is not None
print("[SUCCESS] Gradients successfully flow through all branches and fusion layers.")

Gradient backpropagation check:
  CNN branch earliest conv layer grad:      True (norm: 9.8629e-01)
  ViT branch tokenizer projection grad:     True (norm: 1.4293e+00)
  Fusion projection linear grad:            True (norm: 4.6611e+00)
  Classification head linear grad:          True (norm: 2.4830e+00)
[SUCCESS] Gradients successfully flow through all branches and fusion layers.


## 8. Feature Representation Inspection

Let's visualize the latent feature distributions across the CNN branch, ViT branch, and fused projection for a sample patch:

In [6]:
hybrid_model.eval()
with torch.inference_mode():
    _, feats = hybrid_model(x_batch[:1], return_features=True)

cnn_vec = feats["cnn_features"][0].cpu().numpy()
vit_vec = feats["vit_features"][0].cpu().numpy()
fused_vec = feats["fused_features"][0].cpu().numpy()

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=False)
fig.suptitle("Latent Feature Vectors for a Hyperspectral Patch (Untrained Hybrid)", fontsize=13, fontweight="bold")

axes[0].stem(range(len(cnn_vec)), cnn_vec)
axes[0].set_title(f"3D-CNN Feature Vector (Dim = {len(cnn_vec)})")
axes[0].set_ylabel("Activation")
axes[0].grid(True, alpha=0.3)

axes[1].stem(range(len(vit_vec)), vit_vec)
axes[1].set_title(f"ViT Feature Vector (Dim = {len(vit_vec)})")
axes[1].set_ylabel("Activation")
axes[1].grid(True, alpha=0.3)

axes[2].stem(range(len(fused_vec)), fused_vec)
axes[2].set_title(f"Fused Unified Vector (Dim = {len(fused_vec)})")
axes[2].set_xlabel("Feature Dimension Index")
axes[2].set_ylabel("Activation")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig_dir = PROJECT_ROOT / "outputs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / "hybrid_feature_vectors.png", dpi=150)
plt.show()

print(f"Saved feature vector visualization to {fig_dir / 'hybrid_feature_vectors.png'}")

Saved feature vector visualization to C:\Users\abhis\Desktop\Tetraspectra\outputs\figures\hybrid_feature_vectors.png


C:\Users\abhis\AppData\Local\Temp\ipykernel_17012\2795901873.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Architectural Summary, Limitations & Next Steps

### Architecture Summary

| Model Component | Mechanism | Dimension / Parameters |
| :--- | :--- | :--- |
| **Input Patch** | 5D spatial-spectral tensor | $(B, 1, 15, 7, 7)$ |
| **3D-CNN Branch** | 3D convolutions + AdaptiveAvgPool3d | $D_{\text{cnn}} = 32$ (23,352 params) |
| **ViT Branch** | Spatial tokenization + Self-Attention | $D_{\text{vit}} = 64$ (72,272 params) |
| **Concatenation** | Feature-axis concatenation | $(B, 32 + 64) = (B, 96)$ |
| **Fusion Layer** | Linear + LayerNorm + GELU + Dropout | $D_{\text{fusion}} = 64$ (6,336 params) |
| **Classification Head** | Dense linear classifier | $(B, 16)$ (1,040 params) |
| **Total Model** | Unified CNN-ViT Hybrid | **103,000 parameters** |

### Limitations & Honest Assessment
- **Untrained Architecture**: Today's successful forward pass confirms mathematical compatibility, dimensional alignment, and gradient flow. It does **not** evaluate classification performance.
- **Compute Overhead**: The hybrid model contains 103,000 parameters, roughly $4.4\times$ larger than the baseline 3D-CNN (~23k parameters). Training time per epoch will naturally be longer than Day 10.
- **LayerNorm vs. BatchNorm**: `LayerNorm` was selected inside the fusion projection to enable robust single-sample inference without running into `BatchNorm1d` zero-variance crashes on $B=1$.

### Next Step: Day 14
On **Day 14**, we will train this hybrid model end-to-end on the Salinas dataset using the established AdamW training loop, evaluate its validation convergence, and benchmark its metrics against the Day 10 baseline 3D-CNN!